# Guitar Effect Switcher

Line-in audio is routed through the selectable hardware chain:

`Noise Gate -> Overdrive -> Distortion -> EQ -> Reverb`


In [ ]:
import audio_lab_pynq as aud
from IPython.display import display, clear_output

ol = aud.AudioLabOverlay()

def configure_audio_path(volume=0xE7):
    # Line-in AUX -> ADC
    ol.codec.R4_RECORD_MIXER_LEFT_CONTROL_0 = 0x01
    ol.codec.R5_RECORD_MIXER_LEFT_CONTROL_1 = 0x05
    ol.codec.R6_RECORD_MIXER_RIGHT_CONTROL_0 = 0x01
    ol.codec.R7_RECORD_MIXER_RIGHT_CONTROL_1 = 0x05
    ol.codec.R19_ADC_CONTROL = 0x03

    # FPGA serial playback -> DAC -> playback mixers
    ol.codec.R58_SERIAL_INPUT_ROUTE_CONTROL = 0x01
    ol.codec.R59_SERIAL_OUTPUT_ROUTE_CONTROL = 0x01
    ol.codec.R36_DAC_CONTROL_0 = 0x03
    ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x21
    ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x41

    # PYNQ-Z2 output jack path
    ol.codec.R26_PLAYBACK_LR_MIXER_LEFT_LINE_OUTPUT_CONTROL = 0x03
    ol.codec.R27_PLAYBACK_LR_MIXER_RIGHT_LINE_OUTPUT_CONTROL = 0x09
    ol.codec.R31_PLAYBACK_LINE_OUTPUT_LEFT_VOLUME_CONTROL = volume
    ol.codec.R32_PLAYBACK_LINE_OUTPUT_RIGHT_VOLUME_CONTROL = volume

    # Keep headphone output path enabled as well.
    ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = volume
    ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = volume
    ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03

    # Codec clocks/DSP
    ol.codec.R61_DSP_ENABLE = 0x01
    ol.codec.R62_DSP_RUN = 0x01
    ol.codec.R65_CLOCK_ENABLE_0 = 0x7F
    ol.codec.R66_CLOCK_ENABLE_1 = 0x03

configure_audio_path()
print("AudioLab overlay loaded. Output path configured.")


In [ ]:
def apply_effects(
    noise_gate_on=False, noise_gate_threshold=8,
    overdrive_on=False, overdrive_tone=65, overdrive_level=100, overdrive_drive=30,
    distortion_on=False, distortion_tone=65, distortion_level=100, distortion=25,
    eq_on=False, eq_low=100, eq_mid=100, eq_high=100,
    reverb_on=False, reverb_decay=30, reverb_tone=65, reverb_mix=20,
):
    words = ol.set_guitar_effects(
        noise_gate_on=noise_gate_on,
        noise_gate_threshold=noise_gate_threshold,
        overdrive_on=overdrive_on,
        overdrive_tone=overdrive_tone,
        overdrive_level=overdrive_level,
        overdrive_drive=overdrive_drive,
        distortion_on=distortion_on,
        distortion_tone=distortion_tone,
        distortion_level=distortion_level,
        distortion=distortion,
        eq_on=eq_on,
        eq_low=eq_low,
        eq_mid=eq_mid,
        eq_high=eq_high,
        reverb_on=reverb_on,
        reverb_decay=reverb_decay,
        reverb_tone=reverb_tone,
        reverb_mix=reverb_mix,
    )
    active = [
        name for name, on in [
            ("Noise Gate", noise_gate_on),
            ("Overdrive", overdrive_on),
            ("Distortion", distortion_on),
            ("EQ", eq_on),
            ("Reverb", reverb_on),
        ] if on
    ]
    route = " -> ".join(["Line-in"] + active + ["Output"]) if active else "Line-in -> Output"
    return words, route

try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets is not available; call apply_effects(...) manually.")
else:
    style = {"description_width": "120px"}
    slider_layout = widgets.Layout(width="420px")

    noise_gate_on = widgets.Checkbox(value=False, description="Noise Gate", indent=False)
    overdrive_on = widgets.Checkbox(value=False, description="Overdrive", indent=False)
    distortion_on = widgets.Checkbox(value=False, description="Distortion", indent=False)
    eq_on = widgets.Checkbox(value=False, description="EQ", indent=False)
    reverb_on = widgets.Checkbox(value=False, description="Reverb", indent=False)

    gate_threshold = widgets.IntSlider(value=8, min=0, max=100, step=1, description="Gate threshold", style=style, layout=slider_layout)
    od_drive = widgets.IntSlider(value=30, min=0, max=100, step=1, description="OD drive", style=style, layout=slider_layout)
    od_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="OD tone", style=style, layout=slider_layout)
    od_level = widgets.IntSlider(value=100, min=0, max=200, step=1, description="OD level", style=style, layout=slider_layout)
    dist_amount = widgets.IntSlider(value=25, min=0, max=100, step=1, description="Distortion", style=style, layout=slider_layout)
    dist_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="Dist tone", style=style, layout=slider_layout)
    dist_level = widgets.IntSlider(value=100, min=0, max=200, step=1, description="Dist level", style=style, layout=slider_layout)
    eq_low = widgets.IntSlider(value=100, min=0, max=200, step=1, description="EQ low", style=style, layout=slider_layout)
    eq_mid = widgets.IntSlider(value=100, min=0, max=200, step=1, description="EQ mid", style=style, layout=slider_layout)
    eq_high = widgets.IntSlider(value=100, min=0, max=200, step=1, description="EQ high", style=style, layout=slider_layout)
    reverb_decay = widgets.IntSlider(value=30, min=0, max=100, step=1, description="Reverb decay", style=style, layout=slider_layout)
    reverb_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="Reverb tone", style=style, layout=slider_layout)
    reverb_mix = widgets.IntSlider(value=20, min=0, max=100, step=1, description="Reverb mix", style=style, layout=slider_layout)

    auto_apply = widgets.Checkbox(value=True, description="Apply on change", indent=False)
    apply_button = widgets.Button(description="Apply", button_style="primary")
    bypass_button = widgets.Button(description="Bypass")
    crunch_button = widgets.Button(description="Crunch")
    lead_button = widgets.Button(description="Lead")
    space_button = widgets.Button(description="Space")
    output = widgets.Output()

    controls = [
        noise_gate_on, overdrive_on, distortion_on, eq_on, reverb_on,
        gate_threshold, od_drive, od_tone, od_level,
        dist_amount, dist_tone, dist_level,
        eq_low, eq_mid, eq_high,
        reverb_decay, reverb_tone, reverb_mix,
    ]

    def params():
        return dict(
            noise_gate_on=noise_gate_on.value,
            noise_gate_threshold=gate_threshold.value,
            overdrive_on=overdrive_on.value,
            overdrive_tone=od_tone.value,
            overdrive_level=od_level.value,
            overdrive_drive=od_drive.value,
            distortion_on=distortion_on.value,
            distortion_tone=dist_tone.value,
            distortion_level=dist_level.value,
            distortion=dist_amount.value,
            eq_on=eq_on.value,
            eq_low=eq_low.value,
            eq_mid=eq_mid.value,
            eq_high=eq_high.value,
            reverb_on=reverb_on.value,
            reverb_decay=reverb_decay.value,
            reverb_tone=reverb_tone.value,
            reverb_mix=reverb_mix.value,
        )

    def update_output(*_):
        words, route = apply_effects(**params())
        with output:
            clear_output(wait=True)
            print(route)
            for key in ["gate", "overdrive", "distortion", "eq", "reverb"]:
                print("%10s: 0x%08x" % (key, words[key]))

    def on_control_change(change):
        if auto_apply.value and change.get("name") == "value":
            update_output()

    def set_values(values):
        old_auto = auto_apply.value
        auto_apply.value = False
        for widget, value in values.items():
            widget.value = value
        auto_apply.value = old_auto
        update_output()

    def bypass(_):
        set_values({noise_gate_on: False, overdrive_on: False, distortion_on: False, eq_on: False, reverb_on: False})

    def crunch(_):
        set_values({noise_gate_on: True, overdrive_on: True, distortion_on: False, eq_on: True, reverb_on: False, od_drive: 45, od_tone: 70, od_level: 105, eq_low: 105, eq_mid: 115, eq_high: 120})

    def lead(_):
        set_values({noise_gate_on: True, overdrive_on: False, distortion_on: True, eq_on: True, reverb_on: True, dist_amount: 55, dist_tone: 70, dist_level: 95, eq_low: 95, eq_mid: 120, eq_high: 125, reverb_decay: 25, reverb_tone: 70, reverb_mix: 16})

    def space(_):
        set_values({noise_gate_on: True, overdrive_on: True, distortion_on: False, eq_on: True, reverb_on: True, od_drive: 35, od_tone: 60, od_level: 95, eq_low: 100, eq_mid: 95, eq_high: 115, reverb_decay: 45, reverb_tone: 75, reverb_mix: 28})

    for control in controls:
        control.observe(on_control_change, names="value")
    apply_button.on_click(update_output)
    bypass_button.on_click(bypass)
    crunch_button.on_click(crunch)
    lead_button.on_click(lead)
    space_button.on_click(space)

    toggles = widgets.HBox([noise_gate_on, overdrive_on, distortion_on, eq_on, reverb_on])
    presets = widgets.HBox([apply_button, bypass_button, crunch_button, lead_button, space_button, auto_apply])
    accordion = widgets.Accordion(children=[
        widgets.VBox([gate_threshold]),
        widgets.VBox([od_drive, od_tone, od_level]),
        widgets.VBox([dist_amount, dist_tone, dist_level]),
        widgets.VBox([eq_low, eq_mid, eq_high]),
        widgets.VBox([reverb_decay, reverb_tone, reverb_mix]),
    ])
    for i, title in enumerate(["Noise Gate", "Overdrive", "Distortion", "EQ", "Reverb"]):
        accordion.set_title(i, title)

    display(widgets.VBox([toggles, presets, accordion, output]))
    update_output()
